# Chain ladder workbench: input review

This actuary-facing workbench loads custom-column claims data, maps it into Rustuary's canonical triangle fields, and exposes the assumptions and model-run metadata for review.

Reserve calculation is intentionally deferred until the public Python `ChainLadder` binding is implemented in Slice 2.


In [ ]:
from pathlib import Path
import sys

import pyarrow.csv as pv
import yaml


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "bindings" / "python" / "src" / "rustuary").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from the Rustuary repository checkout")


repo = find_repo_root(Path.cwd().resolve())
python_source = repo / "bindings" / "python" / "src"
if str(python_source) not in sys.path:
    sys.path.insert(0, str(python_source))

import rustuary as ry


## Load the review data

The example uses business-facing column names rather than canonical Rustuary names so the mapping step remains visible.


In [ ]:
claims_path = repo / "data" / "examples" / "paid_claims_custom_columns.csv"
claims = pv.read_csv(claims_path)
claims


## Declare mapping assumptions

`portfolio` reads the `segment` source column. Valuation date, measure, and currency are explicit constants. The source amounts are cumulative paid claims developed in months. These assumptions are loaded from the reviewable YAML fixture.


In [ ]:
mapping_path = repo / "contracts" / "examples" / "claims_mapping.yaml"
mapping_document = yaml.safe_load(mapping_path.read_text())
mapping = ry.ClaimsMapping(**mapping_document["claims_mapping"])
mapping


In [ ]:
triangle = ry.Triangle.from_frame(claims, mapping=mapping)
triangle.data


## Review canonical shape and assumptions


In [ ]:
input_review = {
    "row_count": triangle.data.num_rows,
    "columns": triangle.data.column_names,
    "origin_periods": sorted(set(triangle.data["origin_period"].to_pylist())),
    "development_ages": sorted(set(triangle.data["development_age"].to_pylist())),
    "portfolio_ids": sorted(set(triangle.data["portfolio_id"].to_pylist())),
}
input_review


In [ ]:
expected_columns = [
    "portfolio_id",
    "valuation_date",
    "origin_period",
    "development_age",
    "measure",
    "amount",
    "currency",
    "is_cumulative",
]

assert triangle.data.num_rows == 6
assert triangle.data.column_names == expected_columns
assert triangle.data["portfolio_id"].to_pylist() == ["Motor"] * 6
assert triangle.data["is_cumulative"].to_pylist() == [True] * 6
"Canonical input checks passed."


## Capture review metadata

The detached metadata snapshot records the canonical schema version and every mapping choice for later model-run audit persistence.


In [ ]:
model_run_metadata = triangle.model_run_metadata.to_dict()
assert model_run_metadata["canonical_schema"] == "claims_triangle"
assert model_run_metadata["claims_mapping"]["development_unit"] == "months"
model_run_metadata


## Next calculation step

The reviewed `triangle` is ready for the deterministic Rust chain-ladder engine. Once Slice 2 exposes that engine through Python, this workbench will add factor selection, tail assumptions, reserve results, and diagnostics without reimplementing actuarial calculations in notebook cells.
